# Week 7 – "The Price is Right" (makinda)

**Goal:** Better price prediction with an open-source model (QLoRA fine-tuned Llama).

- **Runtime:** Tuned for **free Colab with T4 GPU** (~15GB VRAM). Batch size and train subset are set to avoid OOM.
- **Data:** `ed-donner/items_lite` (Hugging Face). **Models:** pushed to **felixmakinda**. Course benchmarks: Fine-tuned Lite ~65, Fine-tuned Full ~40; we aim to get close with Llama-3.2-3B + QLoRA.
- **Concepts:** Week 7 (QLoRA, prompt data, completion-only loss), Week 2 (APIs), Week 1 Day 5, results.ipynb.

In [ ]:
# Install deps (run in Colab) – trl, wandb for training logs
%pip install -q datasets transformers torch peft bitsandbytes accelerate pydantic trl wandb

In [ ]:
import os
import re
from typing import Optional

from datasets import load_dataset
from huggingface_hub import login
from pydantic import BaseModel

# Config: dataset from ed-donner, uploads to felixmakinda (tuned for free Colab T4)
HF_DATASET_USER = "ed-donner"
DATASET_NAME = "items_lite"
DATASET = f"{HF_DATASET_USER}/{DATASET_NAME}"
HF_USER = "felixmakinda"
MODEL_REPO_ID = f"{HF_USER}/price-llama-qlora-lite"
BASE_MODEL = "meta-llama/Llama-3.1-8B"
MAX_PROMPT_TOKENS = 110
DO_ROUND_PRICE = True
# T4-friendly: smaller subset to fit 15GB VRAM and avoid timeout
TRAIN_SUBSET = 3000
VAL_SUBSET = 300
SEED = 42

In [ ]:
# HF token: Colab Secrets (HF_TOKEN) or env
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Set HF_TOKEN in Colab Secrets or .env")
login(token=hf_token, add_to_git_credential=True)

# Weights & Biases (set WANDB_API_KEY in Colab Secrets or .env)
try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = os.environ.get("WANDB_API_KEY")
USE_WANDB = bool(wandb_key)
if USE_WANDB:
    import wandb
    wandb.login(key=wandb_key)
    os.environ["WANDB_PROJECT"] = "price-is-right-makinda"
    print("W&B logged in – training will be logged to wandb.")
else:
    print("WANDB_API_KEY not set – training logs will not be sent to W&B.")

## Load dataset (ed-donner)

In [ ]:
class Item(BaseModel):
    title: str
    category: str
    price: float
    summary: Optional[str] = None
    prompt: Optional[str] = None
    completion: Optional[str] = None

    def make_prompts(self, tokenizer, max_tokens: int, do_round: bool):
        text = self.summary or self.title
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) > max_tokens:
            summary = tokenizer.decode(tokens[:max_tokens]).rstrip()
        else:
            summary = text
        self.prompt = f"What does this cost to the nearest dollar?\n\n{summary}\n\nPrice is $"
        self.completion = f"{round(self.price)}.00" if do_round else str(self.price)

ds = load_dataset(DATASET)
train = [Item.model_validate(row) for row in ds["train"]]
val = [Item.model_validate(row) for row in ds["validation"]]
test = [Item.model_validate(row) for row in ds["test"]]
print(f"Loaded {len(train)} train, {len(val)} val, {len(test)} test from {DATASET}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
for item in train[:TRAIN_SUBSET] + val[:VAL_SUBSET]:
    item.make_prompts(tokenizer, MAX_PROMPT_TOKENS, DO_ROUND_PRICE)
for item in test:
    item.make_prompts(tokenizer, MAX_PROMPT_TOKENS, False)
train_ready = [i for i in train if i.prompt and i.completion][:TRAIN_SUBSET]
val_ready = [i for i in val if i.prompt and i.completion][:VAL_SUBSET]
print(f"Train ready: {len(train_ready)}, Val ready: {len(val_ready)}")

## Push prompt dataset to felixmakinda

Push the prompt/completion splits to felixmakinda HF account for reuse.

In [ ]:
# Optional: push prompt dataset to felixmakinda
from datasets import DatasetDict, Dataset
prompt_ds = DatasetDict({
     "train": Dataset.from_list([{"prompt": i.prompt, "completion": i.completion} for i in train_ready]),
     "val": Dataset.from_list([{"prompt": i.prompt, "completion": i.completion} for i in val_ready]),
 })
prompt_ds.push_to_hub(f"{HF_USER}/items_prompts_lite_makinda")

## QLoRA fine-tuning (for better price prediction)

Train only on the **completion** (price) using a prompt-completion dataset and `completion_only_loss=True` in SFTConfig (trl ≥0.20 no longer uses DataCollatorForCompletionOnlyLM). Then push to **felixmakinda**.

In [ ]:
# Build HF Dataset with "prompt" and "completion" (trl prompt-completion format; enables completion_only_loss)
from datasets import Dataset as HFDataset

train_ds = HFDataset.from_list([
    {"prompt": i.prompt or "", "completion": i.completion or "", "price": i.price} for i in train_ready
])
val_ds = HFDataset.from_list([
    {"prompt": i.prompt or "", "completion": i.completion or "", "price": i.price} for i in val_ready
])
print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

In [ ]:
# Load 4-bit quantized base model (Colab-friendly)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# completion_only_loss=True in SFTConfig (trl >= 0.20): loss only on completion tokens, not prompt
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Training: T4-friendly (batch 2, seq 128 to fit ~15GB VRAM)
MAX_SEQ_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4
WARMUP_RATIO = 0.03
SAVE_STEPS = 100
EVAL_STEPS = 50  # If T4 OOM: set BATCH_SIZE=1, GRAD_ACCUM=16

sft_config = SFTConfig(
    output_dir="./price_qlora_out",
    max_seq_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_32bit",
    bf16=True,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=2,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=True,
    hub_model_id=MODEL_REPO_ID,
    report_to="wandb" if USE_WANDB else "none",
)

In [ ]:
# Train and push to felixmakinda (no data_collator: prompt-completion + completion_only_loss)
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    args=sft_config,
)
trainer.train()
trainer.push_to_hub(MODEL_REPO_ID)
print(f"Done. Model pushed to {MODEL_REPO_ID}")

## Evaluate fine-tuned model (better price prediction)

Use the model we just trained (or load from `MODEL_REPO_ID` if running eval-only). Compare average error to course baselines (e.g. Fine-tuned Lite ~65, Base Llama ~110).

In [ ]:
def parse_price(text: str) -> float:
    text = (text or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

def evaluate(predict_fn, items, size=200):
    errors = []
    for i, item in enumerate(items):
        if i >= size:
            break
        pred = predict_fn(item)
        guess = parse_price(pred) if isinstance(pred, str) else pred
        errors.append(abs(guess - item.price))
    avg = sum(errors) / len(errors) if errors else 0
    print(f"Average absolute error: ${avg:.2f}")
    return avg

# Predictor from fine-tuned model (run after training)
def make_model_predictor(model, tokenizer, max_new_tokens=15):
    def predict(item):
        inp = tokenizer(item.prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inp, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, do_sample=False)
        text = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        return parse_price(text)
    return predict

# After training: evaluate fine-tuned model (aim for better than base ~110, target ~65 or lower)
EVAL_SIZE = 200
print("Baseline (constant $50):")
evaluate(lambda item: "50.00", test, size=min(EVAL_SIZE, len(test)))
print("\nFine-tuned model:")
ft_predict = make_model_predictor(trainer.model, tokenizer)
evaluate(ft_predict, test, size=min(EVAL_SIZE, len(test)))